In [50]:
import torch

In [51]:
torch.cuda.is_available()

True

In [52]:
torch.cuda.device_count()

1

In [53]:
torch.cuda.get_device_name()

'NVIDIA GeForce RTX 5070 Ti'

In [54]:
torch.cuda.device(0)

### Tensors

In [55]:
import numpy as np
npArr = np.array([[1,2,3], [1, 2, 3]])

tensor = torch.tensor([[[[1., 2.], 
                            [3., 4.]], [[1., 2.], 
                                        [3., 4.]], [[1., 2.], 
                                                    [3., 4.]]], [[[1., 2.], 
                                                                  [3., 4.]], [[1., 2.], 
                                                                              [3., 4.]], [[1., 2.], 
                                                                                          [3., 4.]]]]) 

print(tensor.size())


torch.Size([2, 3, 2, 2])


In [56]:
tensor.to("cuda")

tensor([[[[1., 2.],
          [3., 4.]],

         [[1., 2.],
          [3., 4.]],

         [[1., 2.],
          [3., 4.]]],


        [[[1., 2.],
          [3., 4.]],

         [[1., 2.],
          [3., 4.]],

         [[1., 2.],
          [3., 4.]]]], device='cuda:0')

### Autograd

In [57]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

f = 3 * a**3 - b**2
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

$$f= 3a^3 -b^2$$
$$\frac{\partial f}{\partial a} = 9a^2$$
$$\frac{\partial f}{\partial b} = -2b$$


In [58]:
f.backward(gradient=torch.tensor([1,1]))
print(a.grad)
print(b.grad)

tensor([36., 81.])
tensor([-12.,  -8.])


### Basic NN

In [59]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [60]:
#set up the data that will train and test the NN
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2)
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.fit_transform(X_test)

### Turning the data into Tensors

In [61]:
X_train_scaled_tesnor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tesnor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)   #turns this from a 1d array to a 2d array
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)     #because pytoarch needs 2d arrays?

In [62]:
train_dataset = TensorDataset(X_train_scaled_tesnor, y_train_tensor)

In [63]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) #batch_size will mean 64 examples are shown to the
                                                                      #nn per loop

### Class definition time!!

In [64]:
class BCNet(nn.Module):
    
    def __init__(self):
        super(BCNet, self).__init__()
        
        self.fc1 = nn.Linear(30, 64)          #fc1 is the first "fully connected" layer, which recieves 30 inputs
                                              #and connects those values to 64 neurons with weights as the edges
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)           #we only want one value to be output as a probability so we only 1 
                                              #output neuron
                                              
    def forward(self, x):                     #forward describes what happens when we pass an input to the network
        x = F.relu(self.fc1(x))               #take the input vector/matrix, applies the transformation x*w + b
                                              #relu is the activation function that wraps the sum +bias of all wieghts
                                              #and determines the activation val for each neuron in layer 1
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        
        return x

In [65]:
model = BCNet()

In [68]:
criterion = nn.BCELoss() #binary cross entropy loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [70]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f'Epoch {epoch + 1}: Loss was {running_loss / len(train_loader)}')

Epoch 1: Loss was 0.03932019059235851
Epoch 2: Loss was 0.03736291825771332
Epoch 3: Loss was 0.040478783659636974
Epoch 4: Loss was 0.03552016479273637
Epoch 5: Loss was 0.032144955514619745
Epoch 6: Loss was 0.03220068539182345
Epoch 7: Loss was 0.03825582383821408
Epoch 8: Loss was 0.026878559147007763
Epoch 9: Loss was 0.02640178465905289
Epoch 10: Loss was 0.02526235702680424
Epoch 11: Loss was 0.02447748586225013
Epoch 12: Loss was 0.022015582614888746
Epoch 13: Loss was 0.02131817489862442
Epoch 14: Loss was 0.020889330096542834
Epoch 15: Loss was 0.018865820924596242
Epoch 16: Loss was 0.018308991907785337
Epoch 17: Loss was 0.017199949737793454
Epoch 18: Loss was 0.016243405492665865
Epoch 19: Loss was 0.01602013116547217
Epoch 20: Loss was 0.01488366463939504


### The model is effectively trained, but we need to evalueate it on unseen data

In [73]:
with torch.no_grad():   # it will not calculate gradients when making predictions as to save computational resources
    model.eval()        #puts it into the eval mode, so no updated model upon evaluation
    
    preds = model(X_test_scaled_tesnor)
    loss  = criterion(preds, y_test_tensor).item()
    
    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()
    

In [74]:
accuracy

0.9912280440330505